# wf_v4.5 runs with Llama (diary Nov 15): freeze, end to end, cost test
Code: GitHub `TriDang2025/attribute-extraction`, branch `wf-v4.5` (from `wf-v4.4`), folder `step1a/`. Results in `results/wf_v45/`.

Model: Llama-3.1-8B-Instruct, bf16, vLLM, temperature 0, guided JSON (setting `llama8b`). Runtime: **A100 (40 GB or more)**.

- Cells 1-5: setup (as the wf_v4.4 notebook; cell 4 logs the loading time to `results/wf_v45/model_load.json`).
- Cell 6: data check.
- Cell 7: **W1** Step 4 v4.4 with "other" off on a_llama44 then c1 (dev50 + dev50b, seeds 0-2 + vote), score, freeze check. About 70 calls, 10-20 min (estimate).
- After a crash or disconnect: rerun cells 1, 3, 4, 5 and then the cell you were on. Answered calls are read from the cache.
- Colab Secrets `HF_TOKEN` and `GH_TOKEN`, notebook access on. Never put a token in a cell.
- When the GPU work is done: **Runtime > Disconnect and delete runtime**, so the paid GPU stops.


In [ ]:
#@title 1. Tokens, clone/pull wf-v4.5, git identity
GIT_NAME = "Your Name"  #@param {type:"string"}
GIT_EMAIL = "you@users.noreply.github.com"  #@param {type:"string"}
BRANCH = "wf-v4.5"  #@param {type:"string"}
import os, subprocess
from google.colab import userdata
from huggingface_hub import login

hf_token = (userdata.get("HF_TOKEN") or "").strip()
if hf_token:
    login(token=hf_token)
del hf_token
os.environ["GH_TOKEN"] = (userdata.get("GH_TOKEN") or "").strip()
def run(cmd, **kw): subprocess.run(cmd, check=True, **kw)
run(["git", "config", "--global", "user.name", GIT_NAME])
run(["git", "config", "--global", "user.email", GIT_EMAIL])
run(["git", "config", "--global", "credential.helper",
     '!f() { echo username=token; echo "password=$GH_TOKEN"; }; f'])
REPO = "/content/attribute-extraction"
if not os.path.exists(REPO):
    run(["git", "clone", "-q", "-b", BRANCH, "https://github.com/TriDang2025/attribute-extraction.git", REPO])
else:
    run(["git", "-C", REPO, "checkout", "-q", BRANCH])
    run(["git", "-C", REPO, "pull", "-q", "--rebase", "--autostash"])
%cd /content/attribute-extraction/step1a
!git log -1 --format="code at %h (%an, %ar): %s"

**Cell 2 note:** the install prints dependency-resolver warnings; ignore them. Then **Runtime > Restart session**, rerun cell 1 and skip cell 2.
gliner and sentence-transformers are installed with `--no-deps` so they do not replace the torch / transformers that vLLM needs. Cell 4 checks that they load.

In [ ]:
#@title 2. Install (pinned; a few minutes; then restart the session and skip this cell)
!pip install -q vllm==0.33.0
!pip install -q --no-deps gliner==0.2.29 sentence-transformers==5.6.0
!pip install -q sentencepiece

In [ ]:
#@title 3. Model setting (config/wf_v41_models.json) and GPU check
SETTING = "llama8b"  #@param ["llama8b"]
import sys
sys.path.insert(0, "/content/attribute-extraction/step1a")
from s1a import nov16_v41 as NV
SET = NV.model_setting(SETTING)
print(SETTING, SET)
!nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv,noheader
import torch
assert torch.cuda.is_available(), "no GPU: change the runtime type to A100"
gb = torch.cuda.get_device_properties(0).total_memory / 2**30
print(f"GPU memory {gb:.1f} GiB")
assert gb >= SET["min_gpu_gb"], f"only {gb:.1f} GiB: {SETTING} needs {SET['min_gpu_gb']} GiB (an A100)"

In [ ]:
#@title 4. Load the model (gpu_memory_utilization leaves room for bge-m3)
MODEL, REVISION = SET["model"], SET["revision"]   # from the setting of cell 3
GPU_UTIL = 0.80  #@param {type:"number"}
MAX_LEN = 32768
import vllm
from vllm import LLM, SamplingParams
print("vllm", vllm.__version__, "| model", MODEL, "| revision", REVISION)
import time
_t0 = time.time()
llm = LLM(model=MODEL, revision=REVISION, tokenizer_revision=REVISION, quantization=SET["quantization"],
          max_model_len=MAX_LEN, gpu_memory_utilization=GPU_UTIL)
from s1a import log_v44 as L44
print("model load:", L44.save_load_time("results/wf_v45/model_load.json", time.time() - _t0, MODEL, REVISION))


In [ ]:
#@title 5. Backend: temperature 0, guided JSON per job schema, batches, cache per job (s1a/log_v44.VllmBackend44)
BATCH = 64  #@param {type:"integer"}
def sampling(max_tokens, schema):
    kw = dict(temperature=0, max_tokens=max_tokens)
    if schema is not None:
        try:
            from vllm.sampling_params import StructuredOutputsParams
            kw["structured_outputs"] = StructuredOutputsParams(json=schema)
        except ImportError:
            from vllm.sampling_params import GuidedDecodingParams
            kw["guided_decoding"] = GuidedDecodingParams(json=schema)
    return SamplingParams(**kw)
from s1a import log_v44 as L44
backend = L44.VllmBackend44(llm, MODEL, REVISION, sampling, max_len=MAX_LEN, batch=BATCH, vllm_version=vllm.__version__)
backend.model_id

In [ ]:
#@title 6. Data check: bge-m3 vectors, GT levels, frozen prompts, v4.3 split jobs
from pathlib import Path
need = ["results/graph_v22_dev50/spans/vectors.npz", "results/graph_v22_dev50b/spans/vectors.npz",
        "results/loop1_v35/gt_levels.csv", "results/loop1_v37/dev50b/gt/gt_levels.csv",
        "results/wf_v44/t4/a_llama44/dev50/a.columns.json", "results/wf_v44/t4/a_llama44/dev50b/a.columns.json",
        "config/wf_v44_step3.json", "config/wf_v44_step4.json"]
missing = [p for p in need if not Path(p).exists()]
print("missing:", missing or "none")
assert not missing, "files missing: stop and tell Claude"


In [ ]:
#@title 7. W1: Step 4 v4.4 with "other" off (a_llama44, c1), score, freeze check -> results/wf_v45/w1
import sys, subprocess
sys.path.insert(0, "/content/attribute-extraction/step1a/scripts")
import w1_step4_v45 as W1
for inp in W1.INPUTS:
    print(inp, W1.run(backend, inputs=(inp,))[inp])
    p = subprocess.run(["bash", "colab/sync.sh", "push-results", f"wf_v45 W1 step4 other off {inp}", "wf_v45/w1"], capture_output=True, text=True)
    print(p.stdout.strip() or p.stderr[-500:])
W1.T6.score(W1.OUT)
print(W1.write_freeze(W1.OUT))
p = subprocess.run(["bash", "colab/sync.sh", "push-results", "wf_v45 W1 scores", "wf_v45"], capture_output=True, text=True)
print(p.stdout.strip() or p.stderr[-500:])
print("W1 done: disconnect the runtime, then tell Claude 'W1 done'")
